# Home Price Estimator: run the project in Google Colab

This notebook runs the project (Ames house prices) from start to finish in Colab.

**Before you start**
1. Have the project ZIP (`Eligible-Group-17-Assignment.zip`) ready.
2. Have the Kaggle file `train.csv` ready (from the competition "House Prices: Advanced Regression Techniques"), or a `kaggle.json` API key.
3. Run the cells from top to bottom (**Runtime → Run all** also works, but you will be asked to upload files in steps 1 and 2).

**How long it takes:** setup about 1 minute; final training and tests about 2 minutes; the optional model-selection experiments (step 5) about 10 to 20 minutes.
The results of those experiments are already in the zip, so you can skip step 5.

*Note: this notebook was prepared outside Colab and has not been run inside it. If a cell fails, read the message under "If something goes wrong" at the bottom.*

## 1. Upload and unpack the project

In [ ]:
import glob, os, zipfile
from google.colab import files

# Option A (default): upload the zip from your computer.
# Option B: set USE_DRIVE = True and put the zip in your Google Drive (path below).
USE_DRIVE = False
DRIVE_ZIP = "/content/drive/MyDrive/Eligible-Group-17-Assignment.zip"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    zip_path = DRIVE_ZIP
else:
    up = files.upload()                       # choose the project zip
    zip_path = next(n for n in up if n.endswith(".zip"))

with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
    top = z.namelist()[0].split("/")[0]      # the project folder inside the zip
os.chdir(f"/content/{top}")
print("Working folder:", os.getcwd())
print(sorted(os.listdir(".")))

## 2. Get the data

Choose **one** option. The file must end up as `data/train.csv`.

In [ ]:
DATA_SOURCE = "upload"      # "upload" (your own train.csv) or "kaggle" (needs kaggle.json and accepted competition rules)

import shutil
os.makedirs("data", exist_ok=True)
if DATA_SOURCE == "upload":
    up = files.upload()                       # choose train.csv
    name = next(n for n in up if n.lower().endswith(".csv"))
    shutil.move(name, "data/train.csv")
else:
    up = files.upload()                       # choose kaggle.json (Kaggle > Settings > Create New Token)
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip install -q kaggle
    !kaggle competitions download -c house-prices-advanced-regression-techniques -p /content/kaggle_data
    with zipfile.ZipFile("/content/kaggle_data/house-prices-advanced-regression-techniques.zip") as z:
        z.extract("train.csv", "data")

import pandas as pd
d = pd.read_csv("data/train.csv")
print(d.shape)
assert d.shape == (1460, 81), "Unexpected shape: this should be Kaggle's train.csv (1460 rows, 81 columns)"
print("OK: data loaded")

## 3. Install the extra packages

In [ ]:
!pip install -q lightgbm tabulate nbformat nbclient nbconvert streamlit fastapi httpx
import sklearn, pandas, numpy, lightgbm
print("scikit-learn", sklearn.__version__, "| pandas", pandas.__version__, "| numpy", numpy.__version__, "| lightgbm", lightgbm.__version__)

The project was built with pandas 3.0, scikit-learn 1.8 and LightGBM 4.7 (see `requirements.txt`). Colab may have other versions. The code is written to work with
newer pandas and scikit-learn, and because you retrain the model in Colab, the saved model and the code always use the same versions.
Exact numbers can differ slightly with other versions.

## 4. (Optional) Look at the data first

A quick look at the two things the report is built around: the skewed price, and how many "missing" values are really "no such feature".

In [ ]:
import numpy as np, matplotlib.pyplot as plt
raw = pd.read_csv("data/train.csv")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(raw.SalePrice / 1e3, bins=40); ax[0].set(title="Sale price (USD thousands)")
ax[1].hist(np.log1p(raw.SalePrice), bins=40); ax[1].set(title="log(1 + price)")
plt.show()
na = raw.isna().mean().sort_values(ascending=False).head(10) * 100
print(na.round(1).to_string())

## 5. (Optional) Model-selection experiments

Cross-validation only (the test set is not used). It compares the baselines and models, tunes settings and writes `reports/*_log.csv` and `models/best_params.json`.
**This takes about 10 to 20 minutes.** The finished results are already in the zip, so skip this cell unless you want to reproduce them.

In [ ]:
RUN_EXPERIMENTS = False     # set to True to re-run the experiments
if RUN_EXPERIMENTS:
    !python experiments.py
else:
    print("Skipped. Using the saved results in reports/ and models/best_params.json")

## 6. Train the final model and score the test set once

Fits on the 80% training split, scores the 20% test split **once**, analyses the errors, then refits on all cleaned data and saves the model.

In [ ]:
!python train.py

In [ ]:
import json
m = json.load(open("reports/metrics.json"))
a = m["targets_vs_achieved"]
print(f"RMSLE   {a['rmsle']['holdout']:.3f}   (target {a['rmsle']['target']})")
print(f"MAE     ${a['mae_usd']['holdout']:,.0f}   (target under ${a['mae_usd']['target']:,})")
print(f"R²      {a['r2']['holdout']:.3f}   (target at least {a['r2']['target']})")
print(f"Within 15%  {a['within_15pct']['holdout']*100:.1f}%   (target at least {a['within_15pct']['target']*100:.0f}%)")
from IPython.display import Image
Image("reports/figures/08_test_results.png")

## 7. Test the whole system and run the unit tests

In [ ]:
!python system_tests.py 2>&1 | tail -40
!python -m pytest tests -q -p no:cacheprovider 2>&1 | tail -5

## 8. Try the estimator

Change the form fields on the right of the next cell (they re-run automatically). First run the cell below to load the model.

In [ ]:
import joblib, json, sys
sys.path.insert(0, os.getcwd())
import housing as h
model = joblib.load("models/price_model.joblib")
defaults = json.load(open("models/defaults.json"))
RMSLE = json.load(open("reports/metrics.json"))["holdout_rmsle"]
print("Model loaded.")

In [ ]:
#@title Describe the home { run: "auto" }
Neighborhood = "NAmes" #@param ["Blmngtn", "Blueste", "BrDale", "BrkSide", "ClearCr", "CollgCr", "Crawfor", "Edwards", "Gilbert", "IDOTRR", "MeadowV", "Mitchel", "NAmes", "NPkVill", "NWAmes", "NoRidge", "NridgHt", "OldTown", "SWISU", "Sawyer", "SawyerW", "Somerst", "StoneBr", "Timber", "Veenker"]
GrLivArea = 1500 #@param {type:"number"}
Stories = 1 #@param [1, 2] {type:"raw"}
LotArea = 9000 #@param {type:"number"}
TotalBsmtSF = 900 #@param {type:"number"}
OverallQual = 6 #@param {type:"slider", min:1, max:10, step:1}
OverallCond = 5 #@param {type:"slider", min:1, max:10, step:1}
YearBuilt = 1975 #@param {type:"integer"}
YearRemodAdd = 1990 #@param {type:"integer"}
GarageCars = 2 #@param [0, 1, 2, 3, 4] {type:"raw"}
FullBath = 2 #@param [0, 1, 2, 3, 4] {type:"raw"}
HalfBath = 0 #@param [0, 1, 2] {type:"raw"}
KitchenQual = "TA" #@param ["Ex", "Gd", "TA", "Fa", "Po"]

inputs = dict(Neighborhood=Neighborhood, GrLivArea=GrLivArea, Stories=Stories, LotArea=LotArea, TotalBsmtSF=TotalBsmtSF,
              OverallQual=OverallQual, OverallCond=OverallCond, YearBuilt=YearBuilt, YearRemodAdd=YearRemodAdd,
              GarageCars=GarageCars, FullBath=FullBath, HalfBath=HalfBath, KitchenQual=KitchenQual)
out = h.safe_estimate(model, defaults, inputs, RMSLE)
if not out["ok"]:
    print("Cannot estimate yet:")
    for e in out["errors"]:
        print("  -", e)
else:
    r = out["result"]
    print(f"Estimated price: ${r['price']:,.0f}")
    print(f"Typical range:   ${r['low']:,.0f} to ${r['high']:,.0f}  (not a guarantee; about 1 in 4 homes fall outside)")
    for w in out["warnings"]:
        print("Warning:", w)
    print("Main drivers (compared with an average Ames home):")
    for d in r["drivers"]:
        src = "from your inputs" if d["from_user"] else "assumed, typical for the neighbourhood"
        print(f"  {d['label']:<24}{d['effect_pct']:+6.1f}%   ({src})")

## 9. (Optional) Rebuild the report and download your results

In [ ]:
!python build_report.py 2>&1 | grep -v -i warn | tail -3

In [ ]:
import shutil
project = os.path.basename(os.getcwd())
shutil.make_archive("/content/project_results", "zip", "/content", project)
files.download("/content/project_results.zip")

## If something goes wrong

| Problem | What to do |
| --- | --- |
| `ModuleNotFoundError` for a package | Run the install cell in step 3 again, then **Runtime → Restart session** and run from step 1 (the files in `/content` are kept unless the runtime was reset) |
| Data shape is not (1460, 81) | You uploaded the wrong file. Use `train.csv` from the Kaggle competition, not `test.csv` |
| Errors that mention a pandas or scikit-learn version | Install the versions the project was built with, then restart the session: `!pip install -q pandas==3.0.2 numpy==2.4.4 scikit-learn==1.8.0 lightgbm==4.7.0` |
| Files disappeared | Colab sessions reset after being idle. Start again from step 1, or save your work to Google Drive |
| Kaggle download says 403 | Open the competition page on Kaggle and click "Join Competition" / accept the rules, then retry |